In [0]:
import dlt
import pyspark.sql.functions as F

catalog = "meridian_tNN"
volume_path = f"/Volumes/{catalog}/landing/raw_files"
entities = ["store_returns", "reason", "store", "item", "customer", "store_sales"]

def generate_bronze_table(table_name):
    @dlt.table(
        name=f"bronze_{table_name}",
        comment=f"Append-only raw layer for {table_name}"
    )
    def create_table():
        return (
            spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "parquet")
            .option("cloudFiles.schemaLocation", f"{volume_path}/_schema_{table_name}")
            .load(f"{volume_path}/{table_name}/*")
            .select(
                "*",
                F.col("_metadata.file_path").alias("source_file_path"),
                F.current_timestamp().alias("ingestion_time"),
                F.regexp_extract(F.col("_metadata.file_path"), r"batch_(\d+)", 1).cast("int").alias("batch_number")
            )
        )

for entity in entities:
    generate_bronze_table(entity)